# Prepare Fine-Tuning Data with Docling and SDG Hub

This notebook runs from an **in-cluster Workbench**. It converts a document with [Docling](https://github.com/docling-project/docling), generates a synthetic question with [SDG Hub](https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub), and writes a Training Hub / LLaMA-Factory JSONL.

Keep `assets/docling-sdg/` beside this notebook. From a Workbench terminal:

```bash
git clone --depth 1 --filter=blob:none --sparse https://github.com/alauda/aml-docs.git
git -C aml-docs sparse-checkout set --no-cone \
  /docs/en/train/guides/prepare-finetune-data-with-docling-and-sdg.ipynb \
  /docs/en/train/guides/assets/docling-sdg/
cd aml-docs/docs/en/train/guides
```

How-to page: [Prepare Fine-Tuning Data with Docling and SDG Hub](./prepare-finetune-data-with-docling-and-sdg.mdx).


## Install packages

Run once per Workbench. Prefer a venv on the workspace PVC so a restart does not drop the install. Then **Kernel → Change Kernel → Python (Docling + SDG)**.

Air-gapped clusters: upload wheels and install as in [Install a Python Wheel File Offline](../../develop/workbench/create_workbench.mdx).


In [ ]:
import os
import sys
from pathlib import Path

# Persistent venv on the workbench PVC. Override if your workspace root differs.
VENV = Path(os.environ.get("DOCLING_SDG_VENV", "/opt/app-root/src/venvs/docling-sdg"))
if not (VENV / "bin" / "python").exists():
    import venv
    VENV.parent.mkdir(parents=True, exist_ok=True)
    venv.EnvBuilder(with_pip=True).create(VENV)

pip = str(VENV / "bin" / "pip")
python = str(VENV / "bin" / "python")
index = os.environ.get("PIP_INDEX_URL", "")
index_args = ["-i", index] if index else []

import subprocess
subprocess.check_call([pip, "install", "-U", "pip", *index_args])
subprocess.check_call(
    [pip, "install", "docling>=2.70", "sdg-hub>=0.7", "datasets", "ipykernel", *index_args]
)
subprocess.check_call(
    [python, "-m", "ipykernel", "install", "--user", "--name", "docling-sdg",
     "--display-name", "Python (Docling + SDG)"]
)
print("Installed into", VENV)
print("Switch this notebook to kernel: Python (Docling + SDG)")
print("Then re-run from the next cell.")
print("sys.executable =", sys.executable)


## Parameters

`SDG_API_BASE` must be the OpenAI-compatible predictor URL, ending in `/v1`. Resolve it as in [Find your model endpoint](../../ai_applications/deploy_agent_with_agentruntime.mdx#find-your-model-endpoint-and-name).

Set `SDG_MODEL` to a name the teacher lists at `/v1/models`. Use an in-cluster `InferenceService` predictor or any other OpenAI-compatible `/v1` chat API.


In [ ]:
import os
from pathlib import Path

ASSET_DIR = Path(os.environ.get("DOCLING_SDG_ASSET_DIR", "assets/docling-sdg"))
DATA_DIR = Path(os.environ.get("DOCLING_SDG_DATA_DIR", "data/docling-sdg"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(DATA_DIR / ".cache" / "huggingface"))
os.environ.setdefault("DOCLING_CACHE_DIR", str(DATA_DIR / ".cache" / "docling"))
SDG_MODEL = os.environ.get("SDG_MODEL", "qwen3")
SDG_API_BASE = os.environ.get(
    "SDG_API_BASE",
    "http://qwen3-predictor.my-namespace.svc.cluster.local/v1",
)
SDG_API_KEY = os.environ.get("SDG_API_KEY", "not-needed")
SDG_MAX_CONCURRENCY = int(os.environ.get("SDG_MAX_CONCURRENCY", "4"))

print("ASSET_DIR", ASSET_DIR.resolve())
print("DATA_DIR", DATA_DIR.resolve())
print("SDG_MODEL", SDG_MODEL)
print("SDG_API_BASE", SDG_API_BASE)


## Convert a document with Docling

The cell uses the sample Markdown in `assets/docling-sdg/` so the notebook runs without a PDF. Drop a `.pdf` or `.docx` into `DATA_DIR / "source"` to convert a real file. Docling downloads layout models on the first PDF convert; they land under `HF_HOME` / `DOCLING_CACHE_DIR`.


In [ ]:
from pathlib import Path

source_dir = DATA_DIR / "source"
out_dir = DATA_DIR / "converted"
source_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

sample = ASSET_DIR / "sample.md"
if sample.exists() and not (source_dir / "sample.md").exists():
    (source_dir / "sample.md").write_text(sample.read_text())

convertible = [
    p for p in sorted(source_dir.iterdir())
    if p.suffix.lower() in {".pdf", ".docx", ".pptx", ".html", ".md"}
]
if not convertible:
    raise FileNotFoundError(f"Put a document in {source_dir}")

from docling.document_converter import DocumentConverter

converter = DocumentConverter()
markdown_paths = []
for path in convertible:
    if path.suffix.lower() == ".md":
        dest = out_dir / path.name
        dest.write_text(path.read_text())
        markdown_paths.append(dest)
        print("copied", path.name)
        continue
    result = converter.convert(str(path))
    dest = out_dir / f"{path.stem}.md"
    dest.write_text(result.document.export_to_markdown())
    markdown_paths.append(dest)
    print("converted", path.name, "->", dest)

print("documents", [p.name for p in markdown_paths])


## Generate synthetic questions with SDG Hub

The bundled flow (`assets/docling-sdg/flow.yaml`) is the three-block LLM test flow from the Kubeflow Pipelines SDG component: build a prompt, call the teacher, extract the question.

For domain knowledge-tuning (several summaries and Q&A pairs per document), clone [`sdg_hub/examples/knowledge_tuning`](https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub/tree/main/examples/knowledge_tuning) and load flow id `epic-jade-656` instead. That flow needs extra columns and a teacher that can hold high concurrency.


In [ ]:
from datasets import Dataset
from sdg_hub import Flow, FlowRegistry

FlowRegistry.discover_flows()
print("built-in flows (first 8):")
for item in FlowRegistry.list_flows()[:8]:
    print(" ", item.get("id"), item.get("name"))

flow = Flow.from_yaml(str(ASSET_DIR / "flow.yaml"))
reqs = flow.get_dataset_requirements()
print("required columns:", getattr(reqs, "required_columns", None))

flow.set_model_config(model=SDG_MODEL, api_base=SDG_API_BASE, api_key=SDG_API_KEY)

dataset = Dataset.from_dict({
    "document": [p.read_text() for p in markdown_paths],
    "domain": ["internal-docs"] * len(markdown_paths),
})

probe = flow.dry_run(
    dataset,
    sample_size=min(1, len(dataset)),
    enable_time_estimation=True,
    max_concurrency=SDG_MAX_CONCURRENCY,
)
print("dry_run columns", probe["final_dataset"]["columns"])
print("dry_run seconds", round(probe["execution_time_seconds"], 2))

generated = flow.generate(
    dataset,
    max_concurrency=SDG_MAX_CONCURRENCY,
    checkpoint_dir=str(DATA_DIR / "sdg-ckpt"),
    save_freq=50,
)
print("generated rows", len(generated))
print(generated)


## Write Training Hub JSONL

Each line is a chat sample. Point `training_hub.sft(..., data_path=...)` or a LLaMA-Factory dataset at this file. See [Fine-tuning LLMs with Training Hub](./training-hub-fine-tuning.mdx).


In [ ]:
import json

sft_path = DATA_DIR / "sft" / "train.jsonl"
sft_path.parent.mkdir(parents=True, exist_ok=True)

written = 0
with sft_path.open("w") as fh:
    for row in generated:
        question = str(row.get("question") or row.get("raw_question") or "").strip()
        if not question:
            continue
        answer = str(row.get("answer") or row["document"]).strip()
        fh.write(json.dumps({
            "messages": [
                {
                    "role": "system",
                    "content": "You are a helpful assistant for this team's documents.",
                },
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ]
        }, ensure_ascii=False) + "\n")
        written += 1

print(f"wrote {written} samples to {sft_path.resolve()}")
print(sft_path.read_text().splitlines()[0][:240])


## Next steps

- Fine-tune: [Training Hub](./training-hub-fine-tuning.mdx) or [Workbench VolcanoJob](./fine-tuning-using-notebooks.mdx).
- Scale PDF conversion: [CodeFlare SDK](../../develop/distributed_workloads/codeflare-sdk-tutorial.mdx) plus the ODH Ray Data Docling notebook.
- Scale SDG: KFP `sdg_hub` component in `pipelines-components` (beta).
- RAG instead of SFT: [AutoRAG](https://github.com/alauda/aml-docs/tree/master/docs/en/train/guides/optimize-rag-with-autorag.ipynb).
